# MERRA-2 VPD Statistics Processing

This notebook processes hourly MERRA-2 VPD (Vapor Pressure Deficit) data to compute daily afternoon VPD statistics.

**Afternoon window:** 12pm-6pm UTC (hours 12-17)  
**Temporal resolution:** Daily  
**Spatial resolution:** Grid cell level (lat, lon preserved)  
**Spatial extent:** US states only (masked using region_mask.nc)  
**Time period:** 1984-2025

## Metrics Computed Per Day

For each day, we compute VPD statistics across the afternoon window:

### VPD Metrics
1. `vpd_mean` - Mean VPD during afternoon window (kPa)
2. `vpd_min` - Minimum VPD during afternoon window (kPa)
3. `vpd_max` - Maximum VPD during afternoon window (kPa)

## Output

- **Directory:** `processed_vpd/`
- **File pattern:** `vpd_{YYYYMM}.nc`
- **Format:** NetCDF4 with dimensions (time, lat, lon)
- **Variables:** 3 metrics (daily VPD statistics)
- **Fill value:** NaN for non-US areas and missing data

In [1]:
# Standard imports
import xarray as xr
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for imports
sys.path.insert(0, str(Path('../../').resolve()))

# Import configuration and src functions
import config
from src.processing import compute_daily_vpd_stats, process_month

In [2]:
# Configuration from config.py
INPUT_DIR = config.DAILY_DATA_DIR
OUTPUT_DIR = config.PROCESSED_VPD_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Time windows from config
AFTERNOON_HOURS = config.AFTERNOON_HOURS

# Date range
START_YEAR = 1984
END_YEAR = 2025

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

# Metric descriptions for process_month
METRIC_NAMES = [
    'vpd_mean',
    'vpd_min',
    'vpd_max',
]

METRIC_DESCRIPTIONS = {
    'vpd_mean': ('Mean VPD during afternoon window',
                'Daily mean vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
    'vpd_min': ('Minimum VPD during afternoon window',
               'Daily minimum vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
    'vpd_max': ('Maximum VPD during afternoon window',
               'Daily maximum vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
}

Input directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/daily_data
Output directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/processed_vpd


## Processing Functions

The processing functions are imported from `src/processing.py`:
- `compute_daily_vpd_stats()`: Compute afternoon VPD statistics for a single day
- `process_month()`: Process all days in a month and save to NetCDF

See [src/processing.py](../../src/processing.py) for implementation details.

In [3]:
# Create wrapper function for VPD processing
def process_vpd_month(year, month):
    """
    Wrapper function to process VPD statistics for a month.
    Calls the generic process_month() with VPD-specific parameters.
    """
    return process_month(
        year=year,
        month=month,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        compute_stats_fn=lambda date, input_dir: compute_daily_vpd_stats(date, input_dir, AFTERNOON_HOURS),
        metric_names=METRIC_NAMES,
        metric_descriptions=METRIC_DESCRIPTIONS,
        output_prefix='vpd',
        title='MERRA-2 VPD Statistics',
        description='Daily afternoon (12pm-6pm UTC) VPD statistics',
        time_window_str='12-17 (UTC)',
        mask_path=config.MASK_FILE,
        use_int16=False,  # VPD uses float32
        fill_value=None,  # Uses NaN for float32
        skip_existing=True
    )

## Test Single Month

Test processing for a single month to verify the workflow.

In [4]:
# Test with a recent month
process_vpd_month(2023, 6)

Output file vpd_202306.nc already exists. Skipping.


True

## Process All Months (1984-2025)

Process all months in the dataset. This will skip any months that have already been processed.

In [5]:
# Process all years and months
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        try:
            process_vpd_month(year, month)
        except Exception as e:
            print(f"Error processing {year}-{month:02d}: {e}")
            continue

Output file vpd_198401.nc already exists. Skipping.
Output file vpd_198402.nc already exists. Skipping.
Output file vpd_198403.nc already exists. Skipping.
Output file vpd_198404.nc already exists. Skipping.
Output file vpd_198405.nc already exists. Skipping.
Output file vpd_198406.nc already exists. Skipping.
Output file vpd_198407.nc already exists. Skipping.
Output file vpd_198408.nc already exists. Skipping.
Output file vpd_198409.nc already exists. Skipping.
Output file vpd_198410.nc already exists. Skipping.
Output file vpd_198411.nc already exists. Skipping.
Output file vpd_198412.nc already exists. Skipping.
Output file vpd_198501.nc already exists. Skipping.
Output file vpd_198502.nc already exists. Skipping.
Output file vpd_198503.nc already exists. Skipping.
Output file vpd_198504.nc already exists. Skipping.
Output file vpd_198505.nc already exists. Skipping.
Output file vpd_198506.nc already exists. Skipping.
Output file vpd_198507.nc already exists. Skipping.
Output file 

## Verify Output

Quick verification of the output files.

In [6]:
# List all output files
output_files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f"Total output files: {len(output_files)}\n")

if output_files:
    # Inspect most recent file
    latest_file = output_files[-1]
    print(f"Inspecting: {latest_file.name}")
    ds = xr.open_dataset(latest_file)
    print(ds)
    print("\nData variables:")
    print(list(ds.data_vars))
    ds.close()

Total output files: 504

Inspecting: vpd_202512.nc
<xarray.Dataset> Size: 2MB
Dimensions:   (time: 31, lat: 51, lon: 95)
Coordinates:
  * time      (time) datetime64[ns] 248B 2025-12-01 2025-12-02 ... 2025-12-31
  * lat       (lat) float64 408B 24.0 24.5 25.0 25.5 ... 47.5 48.0 48.5 49.0
  * lon       (lon) float64 760B -125.0 -124.4 -123.8 ... -67.5 -66.88 -66.25
Data variables:
    vpd_mean  (time, lat, lon) float32 601kB ...
    vpd_min   (time, lat, lon) float32 601kB ...
    vpd_max   (time, lat, lon) float32 601kB ...
Attributes:
    title:                MERRA-2 VPD Statistics
    description:          Daily afternoon (12pm-6pm UTC) VPD statistics
    source:               MERRA-2 M2T1NXSLV collection
    afternoon_hours:      12-17 (UTC)
    temporal_resolution:  Daily
    spatial_mask:         US states only (state_mask > 0)
    created:              2026-03-23 11:15:07

Data variables:
['vpd_mean', 'vpd_min', 'vpd_max']
